# Week 09 · Social Graphs & Social Network Analysis Fundamentals
**ISB02306504 – Social Media Mining** · Information Systems for Business · Odd Semester 2026/2027

This week adds the network lens: **who connects to whom** (slide 2).

| Part | What we do | Slides |
|---|---|---|
| 0 | Setup | – |
| 1 | Graph fundamentals on the slide example: nodes, edges, direction, weights, attributes, edge list, adjacency | 4–14 |
| 2 | One posts table → five different graphs (mention, reply, repost, hashtag, user–hashtag) | 16–22 |
| 3 | Real social media data: 14,640 tweets to US airlines. Degree, in/out-degree, strength, density, integrated content + network view | 24–34 |
| 4 | A real directed network at scale (email-Eu-core): reciprocity, components, and why density is easy to misuse | 30–34 |
| 5 | Visualisation and export best practices, class activity, homework | 36–41 |

---
## 0. Setup
NetworkX, pandas and matplotlib are preinstalled on Colab.

In [ ]:
import re
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from networkx.algorithms import bipartite
from IPython.display import display

SEED = 42
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9})
BLUE, ORANGE, GREY, GREEN = "#2a78d6", "#eb6834", "#8a8984", "#1baf7a"
print("networkx", nx.__version__, "| pandas", pd.__version__)

---
## 1. Graph fundamentals

A graph is **G = (V, E)**: a set of nodes (vertices) and a set of edges (links). What V and E *mean* is defined by your analytical question, not by the software.

### 1.1 Write the network specification first
Slide 8 asks four questions before drawing anything. Writing the answers down is the single most useful habit in network analysis: two analysts who skip this step will build different graphs from the same data and argue about the results.

In [ ]:
SPEC = {
    "node":      "a user account",
    "edge":      "a reply event: A replied to B",
    "direction": "yes: replier → replied-to",
    "weight":    "number of replies from A to B in the period",
    "period":    "1–30 Oct 2026",
}
for k, v in SPEC.items():
    print(f"{k:>10}: {v}")

### 1.2 The slide example as a directed graph
Edges: A→B, A→C, D→C, E→C, C→B.

In [ ]:
G = nx.DiGraph()
edges = [("A", "B"), ("A", "C"), ("D", "C"), ("E", "C"), ("C", "B")]
G.add_edges_from(edges)

print("nodes:", G.number_of_nodes(), list(G.nodes()))
print("edges:", G.number_of_edges(), list(G.edges()))
print("in-degree :", dict(G.in_degree()))
print("out-degree:", dict(G.out_degree()))
print("degree    :", dict(G.degree()))

Count it by hand first (slide 26): for each node, how many arrows enter and how many leave? Then compare with the code. The degree table from slide 27:

In [ ]:
deg_table = pd.DataFrame({"in-degree": dict(G.in_degree()), "out-degree": dict(G.out_degree()),
                          "total degree": dict(G.degree())}).sort_index()
display(deg_table)

pos = {"A": (0, 1), "B": (2, 1), "C": (1, 0.45), "D": (0, 0), "E": (2, 0)}       # fixed positions as on the slide
fig, ax = plt.subplots(figsize=(4.5, 3))
nx.draw_networkx(G, pos, ax=ax, node_size=900, node_color=BLUE, font_color="white", arrows=True,
                 arrowsize=18, edge_color=GREY, connectionstyle="arc3,rad=0.05")
ax.set_title("Slide example (directed)"); ax.axis("off"); plt.tight_layout(); plt.show()

**Which user appears prominent?** (slide 28) C has the highest total degree and the highest in-degree (most incoming attention); A has the highest out-degree (most outgoing activity). A metric is only useful when its meaning matches the stakeholder's question.

### 1.3 Density
Density compares observed edges *m* with possible edges for *n* nodes:

$$\text{undirected: } \frac{2m}{n(n-1)} \qquad\qquad \text{directed: } \frac{m}{n(n-1)}$$

In [ ]:
n, m = G.number_of_nodes(), G.number_of_edges()
print(f"Directed  : nx.density = {nx.density(G):.2f}   by hand: {m} / ({n}×{n - 1}) = {m / (n * (n - 1)):.2f}")
U = G.to_undirected()
print(f"Undirected: nx.density = {nx.density(U):.2f}   by hand: 2×{U.number_of_edges()} / ({n}×{n - 1}) = "
      f"{2 * U.number_of_edges() / (n * (n - 1)):.2f}")

Same five relationships, two different densities, because a directed graph has twice as many *possible* edges. Write the interpretation like slide 39:

> "The directed reply network has density 0.25, meaning 25% of all possible directed user-to-user links are observed **within the selected time window**."

### 1.4 Undirected graphs
When the relationship is symmetric ("appeared together", hashtag co-occurrence), A — B is the same as B — A (slide 9).

In [ ]:
H = nx.Graph()
H.add_edges_from([("#AI", "#Education"), ("#Education", "#AI"), ("#AI", "#Data")])
print("Edges stored:", list(H.edges()), "→ the duplicate (#Education, #AI) collapsed into the same edge")
print("Is #Education → #AI an edge?", H.has_edge("#Education", "#AI"))

### 1.5 Weighted graphs and strength
Raw social data are **events**: every reply is one row. Weights preserve intensity instead of collapsing repeated interactions into a single line.

In [ ]:
events = pd.DataFrame({
    "source": ["A", "A", "A", "A"],
    "target": ["B", "B", "B", "C"],
    "type":   ["reply"] * 4,
    "time":   pd.to_datetime(["2026-10-02 09:10", "2026-10-02 09:40", "2026-10-05 20:15", "2026-10-07 13:00"]),
})
display(events)

edge_list = (events.groupby(["source", "target"])
             .agg(weight=("time", "size"), first_seen=("time", "min"), last_seen=("time", "max"))
             .reset_index())
display(edge_list)

W = nx.from_pandas_edgelist(edge_list, "source", "target", edge_attr=True, create_using=nx.DiGraph())
print("out-degree of A  :", W.out_degree("A"), "  (distinct people A replied to)")
print("out-strength of A:", W.out_degree("A", weight="weight"), "  (total replies A sent = 3 + 1)")

**Document the weight.** "3" is meaningless unless we know what was counted (slide 10). Here: *number of replies from source to target between 1 and 30 Oct 2026*.

⚠️ **Pitfall: duplicate rows silently disappear.** If you pass the raw events straight into a `DiGraph`, each new row *overwrites* the previous edge between the same pair. No error, no warning:

In [ ]:
naive = nx.from_pandas_edgelist(events, "source", "target", create_using=nx.DiGraph())
print("Events:", len(events), "| edges in naive DiGraph:", naive.number_of_edges(),
      "| weight information:", nx.get_edge_attributes(naive, "weight") or "none")

multi = nx.from_pandas_edgelist(events, "source", "target", edge_attr=["time"], create_using=nx.MultiDiGraph())
print("MultiDiGraph keeps every event:", multi.number_of_edges(), "parallel edges")

Two correct options: **aggregate first** (`groupby` → weight, as above, the usual choice for analysis) or keep every event in a `MultiDiGraph` (useful when each event carries its own timestamp or text).

### 1.6 Edges can carry attributes
Weight, timestamp, interaction type and sentiment can all live on the edge. This is how network structure meets the content analysis of weeks 5–7.

In [ ]:
edge_df = pd.DataFrame({
    "source":    ["A", "A", "D", "E", "C"],
    "target":    ["B", "C", "C", "C", "B"],
    "type":      ["reply", "mention", "reply", "repost", "mention"],
    "weight":    [3, 1, 2, 1, 2],
    "sentiment": ["negative", "neutral", "negative", "positive", "neutral"],
})
display(edge_df)                                           # slide 13: the edge list

G2 = nx.from_pandas_edgelist(edge_df, source="source", target="target", edge_attr=True, create_using=nx.DiGraph())
print("Attributes on A→B:", G2.edges["A", "B"])
neg_in = Counter(t for s, t, a in G2.edges(data=True) if a["sentiment"] == "negative")
print("Negative interactions received:", dict(neg_in))

### 1.7 Three ways to store the same graph  *(slide 14)*

In [ ]:
print("Edge list      :", list(G.edges()))
print("Adjacency list :", {u: list(nbrs) for u, nbrs in G.adjacency()})
print("Adjacency matrix (rows = source, columns = target):")
display(nx.to_pandas_adjacency(G, nodelist=sorted(G.nodes()), dtype=int))

| Format | Best for |
|---|---|
| Edge list | data collection, CSV files, imports |
| Adjacency list | neighbour-centric traversal ("who does A talk to?") |
| Adjacency matrix | matrix-based reasoning; **store it sparse** for real networks |

A dense matrix for 1 million users would need 10¹² cells. Real social graphs are extremely sparse, so use `nx.to_scipy_sparse_array(G)` and never build the dense version.

---
## 2. Same data, different graph  *(slides 16–22)*
One table of posts can produce several graphs, and each graph answers a different question (slide 22). The table below is a small **made-up** example with every field from the slide: author, mentions, hashtags, reply target and repost source.

In [ ]:
posts = pd.DataFrame([
    # post_id, author, text, reply_to, repost_of
    (1, "kevin", "Paket belum sampai @tokoku #delivery #complaint", None, None),
    (2, "vier",  "@kevin sama, punyaku juga telat #delivery", "kevin", None),
    (3, "jolie", "Promo hari ini! #sale #promo", None, None),
    (4, "nico",  "", None, "jolie"),
    (5, "kevin", "@tokoku tolong dicek ya #complaint", None, None),
    (6, "tokoku", "@kevin mohon maaf, kami cek dulu", "kevin", None),
    (7, "vier",  "", None, "jolie"),
    (8, "nico",  "@vier @kevin sabar ya #delivery", "vier", None),
    (9, "jolie", "New arrival #sale #fashion", None, None),
    (10, "kevin", "", None, "jolie"),
], columns=["post_id", "author", "text", "reply_to", "repost_of"])
posts["mentions"] = posts["text"].str.findall(r"@(\w+)")
posts["hashtags"] = posts["text"].str.lower().str.findall(r"#(\w+)")
display(posts)

In [ ]:
def weighted_edges(pairs, directed=True):
    """pairs: iterable of (source, target) events → DataFrame with a weight column."""
    df_ = pd.DataFrame(list(pairs), columns=["source", "target"])
    if not directed:                                          # sort each pair so A–B and B–A are the same edge
        df_[["source", "target"]] = np.sort(df_[["source", "target"]].values, axis=1)
    return df_.groupby(["source", "target"]).size().rename("weight").reset_index()

graphs = {}
graphs["mention"] = nx.from_pandas_edgelist(
    weighted_edges((r.author, m) for r in posts.itertuples() for m in r.mentions),
    "source", "target", edge_attr="weight", create_using=nx.DiGraph())
graphs["reply"] = nx.from_pandas_edgelist(
    weighted_edges((r.author, r.reply_to) for r in posts.itertuples() if r.reply_to),
    "source", "target", edge_attr="weight", create_using=nx.DiGraph())
graphs["repost"] = nx.from_pandas_edgelist(
    weighted_edges((r.author, r.repost_of) for r in posts.itertuples() if r.repost_of),
    "source", "target", edge_attr="weight", create_using=nx.DiGraph())
graphs["hashtag co-occurrence"] = nx.from_pandas_edgelist(
    weighted_edges((a, b) for tags in posts["hashtags"] for a, b in combinations(sorted(set(tags)), 2)),
    "source", "target", edge_attr="weight", create_using=nx.Graph())

UH = nx.Graph()                                              # bipartite: users on one side, hashtags on the other
for r in posts.itertuples():
    for t in set(r.hashtags):
        UH.add_node(r.author, kind="user"); UH.add_node("#" + t, kind="hashtag")
        w = UH.get_edge_data(r.author, "#" + t, {"weight": 0})["weight"]
        UH.add_edge(r.author, "#" + t, weight=w + 1)
graphs["user–hashtag"] = UH

In [ ]:
QUESTION = {"mention": "Who receives mentions?", "reply": "Who talks to whom?",
            "repost": "Whose content is amplified?", "hashtag co-occurrence": "Which themes co-occur?",
            "user–hashtag": "Which users join which themes?"}
summary = pd.DataFrame([{"graph": k, "question": QUESTION[k], "directed": g.is_directed(),
                         "nodes": g.number_of_nodes(), "edges": g.number_of_edges()} for k, g in graphs.items()])
display(summary.set_index("graph"))

fig, axes = plt.subplots(1, 5, figsize=(18, 3.6))
for ax, (name, g) in zip(axes, graphs.items()):
    if name == "user–hashtag":
        users = [n for n, d in g.nodes(data=True) if d["kind"] == "user"]
        p = nx.bipartite_layout(g, users)
        colors = [BLUE if g.nodes[n]["kind"] == "user" else ORANGE for n in g]
    else:
        p = nx.spring_layout(g, seed=SEED)
        colors = ORANGE if "hashtag" in name else BLUE
    widths = [1 + 1.5 * d.get("weight", 1) for _, _, d in g.edges(data=True)]
    arrow_kw = {"arrows": True, "arrowsize": 12} if g.is_directed() else {}
    nx.draw_networkx(g, p, ax=ax, node_color=colors, node_size=500, font_size=7, width=widths,
                     edge_color=GREY, **arrow_kw)
    ax.set_title(f"{name}\n{QUESTION[name]}", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

Ten posts, five graphs, five different "most important" nodes:
- In the **mention** graph, `kevin` and `tokoku` receive attention.
- In the **reply** graph, `kevin` is the one people answer.
- In the **repost** graph, `jolie` is the amplified source, even though she never mentions or replies to anyone.
- In the **hashtag** graph, `#delivery` and `#complaint` form one theme and `#sale` sits in another.
- The **bipartite** graph keeps users and hashtags as different node types (slide 20), which tells you who bridges topic spaces.

| Network | Node | Edge | Direction | Weight? |
|---|---|---|---|---|
| Follower | user | follows | directed | optional |
| Mention | user | mentions | directed | yes |
| Reply | user | replies to | directed | yes |
| Repost | user | reposts from | directed | yes |
| Hashtag co-occurrence | hashtag | same post | undirected | yes |
| User–hashtag | user + hashtag | uses | usually undirected | yes |

A follower graph cannot be built from posts at all: it needs a separate collection of who follows whom. And a follow is a relationship, not proof of attention, trust or influence (slide 16).

---
## 3. Real data: customers tweeting at US airlines  *(slides 16–34)*
**Dataset:** *Twitter US Airline Sentiment* (Crowdflower / Figure Eight, 2015): 14,640 tweets from February 2015 that mention six US airlines, each labelled with sentiment and, for negative tweets, a reason. We read the copy hosted in a public course repository directly into pandas (the zip is decompressed automatically).

**Business question:** *Which airlines receive the most customer attention, how much of it is negative, and do customers also talk to each other?*

In [ ]:
TWEETS_URL = "https://github.com/albahnsen/AppliedDeepLearningClass/raw/master/datasets/Tweets.zip"
tweets = pd.read_csv(TWEETS_URL)
print(tweets.shape)
display(tweets[["tweet_id", "name", "airline", "airline_sentiment", "negativereason", "text", "tweet_created"]].head(4))

### 3.1 Check and prepare the raw table
Before any graph: duplicates, time range, and what the text actually contains.

In [ ]:
print("Duplicate tweet_id rows:", tweets["tweet_id"].duplicated().sum())
tweets = tweets.drop_duplicates("tweet_id").copy()
tweets["created"] = pd.to_datetime(tweets["tweet_created"].str[:19])          # local time as recorded (UTC-8)
tweets["author"] = tweets["name"].str.lower()
tweets["mentions"] = tweets["text"].str.lower().str.findall(r"@(\w+)")
tweets["hashtags"] = tweets["text"].str.lower().str.findall(r"#(\w+)")

print(f"Tweets after de-duplication: {len(tweets):,} | authors: {tweets['author'].nunique():,}")
print("Period:", tweets["created"].min(), "→", tweets["created"].max())
print("Mentions per tweet:", tweets["mentions"].str.len().value_counts().sort_index().to_dict())
print(f"Tweets with ≥1 hashtag: {(tweets['hashtags'].str.len() > 0).mean():.1%}")
print("Retweets ('RT @…'):", tweets["text"].str.startswith("RT @").sum())

Three findings that shape what we can build:
- **Every tweet has at least one @mention** (that is how the data was collected: tweets mentioning an airline). A mention network is natural.
- **Only a handful of retweets** were collected, so a repost network is not possible with this data. *Which graph you can build depends on how the data was collected.*
- Usernames and hashtags are **lower-cased**, otherwise `@United` and `@united` would become two nodes.

Network specification, written before building (slide 8):

In [ ]:
SPEC_MENTION = {
    "node":      "a Twitter account (customer or airline), lower-cased",
    "edge":      "author → account mentioned in the author's tweet",
    "direction": "yes: mentioner → mentioned",
    "weight":    "number of tweets in which the author mentioned that account",
    "attributes": "share of negative tweets, first/last time seen",
    "period":    f"{tweets['created'].min():%d %b %Y} – {tweets['created'].max():%d %b %Y}",
    "collection": "tweets that mention one of six US airlines (not a random sample of Twitter)",
}
for k, v in SPEC_MENTION.items():
    print(f"{k:>10}: {v}")

### 3.2 Build the mention network: events → weighted edge list → graph

In [ ]:
mention_events = (tweets[["author", "mentions", "airline_sentiment", "created"]]
                  .explode("mentions").dropna(subset=["mentions"])
                  .rename(columns={"author": "source", "mentions": "target"}))
self_mentions = (mention_events["source"] == mention_events["target"]).sum()
mention_events = mention_events[mention_events["source"] != mention_events["target"]]
print(f"Mention events: {len(mention_events):,} (removed {self_mentions} self-mentions)")

mention_edges = (mention_events.assign(neg=mention_events["airline_sentiment"].eq("negative"))
                 .groupby(["source", "target"])
                 .agg(weight=("neg", "size"), negative_share=("neg", "mean"),
                      first_seen=("created", "min"), last_seen=("created", "max"))
                 .reset_index())
display(mention_edges.sort_values("weight", ascending=False).head(6))

M = nx.from_pandas_edgelist(mention_edges, "source", "target", edge_attr=True, create_using=nx.DiGraph())
print(f"Mention network: {M.number_of_nodes():,} nodes, {M.number_of_edges():,} edges, density {nx.density(M):.6f}")

The density is tiny: of all the directed links that *could* exist between these accounts, almost none do. That is normal for social networks, and it says nothing about which account matters (slide 34).

### 3.3 In-degree, out-degree and strength  *(slides 24–29)*

In [ ]:
nodes = pd.DataFrame({
    "in-degree":    dict(M.in_degree()),                  # distinct accounts that mention this account
    "in-strength":  dict(M.in_degree(weight="weight")),   # total mentions received
    "out-degree":   dict(M.out_degree()),                 # distinct accounts this account mentions
    "out-strength": dict(M.out_degree(weight="weight")),  # total mentions sent
})
print("Most mentioned (in-degree):")
display(nodes.sort_values("in-degree", ascending=False).head(8))
print("Most active mentioners (out-strength):")
display(nodes.sort_values("out-strength", ascending=False).head(8))

Read the two tables as two different questions (slide 28):
- **In-degree**: the six airline accounts dominate. This is partly *by construction*: the data was collected by searching for them.
- **In-degree vs in-strength**: in-strength counts repeated mentions from the same person, so the gap shows how much attention comes from repeat complainers.
- **Out-strength** highlights accounts that mention airlines very often. Some are customers stuck in a long problem; others are news or "spotter" accounts that post about airlines professionally. Neither is "influence".

### 3.4 Degree distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (col, color) in zip(axes, [("in-degree", ORANGE), ("out-degree", BLUE)]):
    counts = nodes[col][nodes[col] > 0].value_counts().sort_index()
    ax.scatter(counts.index, counts.values, color=color, s=18)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(f"{col} (log)"); ax.set_ylabel("number of accounts (log)"); ax.set_title(f"{col} distribution")
plt.tight_layout(); plt.show()
print("Accounts never mentioned (in-degree 0):", (nodes["in-degree"] == 0).sum())
print("Median out-degree:", nodes.loc[nodes["out-degree"] > 0, "out-degree"].median())

Most accounts mention one or two others and are never mentioned themselves; a tiny number of accounts receive almost all mentions. This **heavy-tailed** shape is typical of social media. It is also why the *average* degree is a poor summary: report the median, the maximum and the distribution.

### 3.5 Integrated lens: network + sentiment  *(slides 2 and 11)*
Edge attributes let us ask "who receives attention, and what kind?"

In [ ]:
AIRLINE_ACCOUNTS = ["united", "usairways", "americanair", "southwestair", "jetblue", "virginamerica"]
air = pd.DataFrame({
    "customers mentioning (in-degree)": nodes.loc[AIRLINE_ACCOUNTS, "in-degree"],
    "mentions received (in-strength)": nodes.loc[AIRLINE_ACCOUNTS, "in-strength"],
    "negative share": [np.average([M.edges[u, a]["negative_share"] for u in M.predecessors(a)],
                                  weights=[M.edges[u, a]["weight"] for u in M.predecessors(a)])
                       for a in AIRLINE_ACCOUNTS],
})
air["mentions per customer"] = air["mentions received (in-strength)"] / air["customers mentioning (in-degree)"]
display(air.sort_values("mentions received (in-strength)", ascending=False)
        .style.format({"negative share": "{:.0%}", "mentions per customer": "{:.2f}"}))

top_reasons = (tweets[tweets["airline_sentiment"] == "negative"]
               .groupby("airline")["negativereason"].agg(lambda s: s.value_counts().index[0]))
print("Most common negative reason per airline:", top_reasons.to_dict())

This combines the two lenses: the network tells us **how much** attention each airline receives and from **how many** distinct customers; the edge attribute tells us **what kind** of attention it is. An airline with high in-degree and a high negative share has a wide, unhappy audience. A high *mentions per customer* value means the same customers come back again and again, which often signals unresolved cases.

### 3.6 Remove the hubs: do customers talk to each other?
The airline accounts dominate everything. Removing them reveals the customer-to-customer layer.

In [ ]:
OTHER_AIRLINES = {"delta", "deltaassist", "virginatlantic", "spiritairlines", "aircanada", "alaskaair", "staralliance"}
customer = M.subgraph([n for n in M if n not in set(AIRLINE_ACCOUNTS) | OTHER_AIRLINES]).copy()
customer.remove_nodes_from([n for n, d in customer.degree() if d == 0])
weak = sorted(nx.weakly_connected_components(customer), key=len, reverse=True)
print(f"Customer-to-customer layer: {customer.number_of_nodes():,} accounts, {customer.number_of_edges():,} edges")
print(f"Weakly connected components: {len(weak):,}; largest has {len(weak[0])} accounts; "
      f"{sum(len(c) == 2 for c in weak):,} are isolated pairs")
print("Most mentioned non-airline accounts:",
      sorted(dict(customer.in_degree(weight="weight")).items(), key=lambda kv: -kv[1])[:8])

In [ ]:
largest = customer.subgraph(weak[0]).copy()
fig, ax = plt.subplots(figsize=(9, 6))
p = nx.spring_layout(largest.to_undirected(), seed=SEED, k=0.3, weight=None, iterations=200)
indeg = dict(largest.in_degree(weight="weight"))
nx.draw_networkx_edges(largest, p, ax=ax, edge_color=GREY, alpha=0.6, arrows=True, arrowsize=8)
nx.draw_networkx_nodes(largest, p, ax=ax, node_size=[20 + 25 * indeg[n] for n in largest],
                       node_color=[ORANGE if indeg[n] >= 3 else BLUE for n in largest], alpha=0.85)
nx.draw_networkx_labels(largest, p, labels={n: n for n in largest if indeg[n] >= 3}, font_size=7, ax=ax)
ax.set_title(f"Largest customer-to-customer component ({largest.number_of_nodes()} accounts; size = mentions received, labelled if ≥ 3)")
ax.axis("off"); plt.tight_layout(); plt.show()

Once the airlines are gone, the network falls apart into many tiny pieces: customers mostly talk *at* airlines, not *with* each other. The largest cluster forms around **airports and media accounts** that customers pull into their complaints (`@phlairport`, `@dfwairport`, `@usatoday`). Outside it, the most mentioned non-airline account is a band promoting a tour with one airline. These are what slide 2 calls interaction clusters.

### 3.7 Reciprocity: does anyone answer?  *(slide 16)*
Reciprocity is the share of directed edges that go both ways (A→B *and* B→A).

In [ ]:
print(f"Reciprocity of the mention network: {nx.reciprocity(M):.3f}")
print("Mentions FROM the airline accounts in this data:",
      {a: M.out_degree(a) for a in AIRLINE_ACCOUNTS})

Almost zero, and the airline accounts have (almost) no outgoing edges. That does **not** mean airlines never reply. It means the dataset contains tweets *to* airlines and not the airlines' own tweets. A network can only show the relationships that were collected: always state the collection rule next to any network result.

### 3.8 Time matters: one graph per day  *(slide 8: "What period is represented?")*

In [ ]:
daily = []
for day, grp in mention_events.groupby(mention_events["created"].dt.date):
    g = nx.from_pandas_edgelist(grp, "source", "target", create_using=nx.DiGraph())
    daily.append({"day": pd.Timestamp(day), "tweets": len(grp), "nodes": g.number_of_nodes(),
                  "edges": g.number_of_edges(), "density": nx.density(g)})
daily = pd.DataFrame(daily).set_index("day")
display(daily.style.format({"density": "{:.5f}", "tweets": "{:,}", "nodes": "{:,}", "edges": "{:,}"}))

The daily graphs differ a lot in size. Notice that **density goes up when the graph gets smaller**: fewer nodes means far fewer *possible* edges. Comparing densities of graphs with different sizes is the first misuse on slide 34.

### 3.9 Hashtag co-occurrence network  *(slide 19)*
Node = hashtag; undirected edge = two hashtags in the same tweet; weight = number of tweets.

In [ ]:
pairs = [tuple(sorted(p)) for tags in tweets["hashtags"] for p in combinations(set(tags), 2)]
hashtag_edges = pd.DataFrame(Counter(pairs).items(), columns=["pair", "weight"])
hashtag_edges[["source", "target"]] = pd.DataFrame(hashtag_edges["pair"].tolist(), index=hashtag_edges.index)
HT = nx.from_pandas_edgelist(hashtag_edges, "source", "target", edge_attr="weight")
tag_count = Counter(t for tags in tweets["hashtags"] for t in set(tags))
nx.set_node_attributes(HT, {t: tag_count[t] for t in HT}, "tweets")

print(f"Hashtag graph: {HT.number_of_nodes()} hashtags, {HT.number_of_edges()} edges, density {nx.density(HT):.4f}")
strength = pd.Series(dict(HT.degree(weight="weight"))).sort_values(ascending=False)
display(pd.DataFrame({"degree": pd.Series(dict(HT.degree())), "strength": strength}).loc[strength.index[:10]])
print("Strongest pairs:", hashtag_edges.nlargest(8, "weight")[["source", "target", "weight"]].values.tolist())

In [ ]:
core = nx.Graph([(u, v, d) for u, v, d in HT.edges(data=True) if d["weight"] >= 2])   # keep repeated pairs only
fig, ax = plt.subplots(figsize=(10, 6.5))
p = nx.spring_layout(core, seed=SEED, k=0.6, weight="weight")
nx.draw_networkx_edges(core, p, ax=ax, width=[0.5 + 0.8 * d["weight"] for _, _, d in core.edges(data=True)],
                       edge_color=GREY, alpha=0.7)
nx.draw_networkx_nodes(core, p, ax=ax, node_size=[15 + 4 * HT.nodes[n]["tweets"] for n in core],
                       node_color=ORANGE, alpha=0.85)
nx.draw_networkx_labels(core, p, font_size=7, ax=ax)
ax.set_title("Hashtag co-occurrence (pairs seen in ≥ 2 tweets; edge width = weight)"); ax.axis("off")
plt.tight_layout(); plt.show()

Complaint tags (`#fail`, `#customerservice`, `#nocustomerservice`, `#badmgmt`) cluster together, aviation-enthusiast tags (`#avgeek`, `#b737`, `#newark`) form their own group, and airline-name hashtags such as `#jetblue` link clusters together. Filtering out pairs seen only once is a **documented** choice: write the threshold in the figure title.

### 3.10 User–hashtag bipartite network  *(slide 20)*

In [ ]:
uh_pairs = tweets[["author", "hashtags"]].explode("hashtags").dropna()
uh_edges = uh_pairs.groupby(["author", "hashtags"]).size().rename("weight").reset_index()
uh_edges["hashtags"] = "#" + uh_edges["hashtags"]                       # avoid a user and a tag with the same name

B = nx.Graph()
B.add_nodes_from(uh_edges["author"].unique(), bipartite="user")
B.add_nodes_from(uh_edges["hashtags"].unique(), bipartite="hashtag")
B.add_weighted_edges_from(uh_edges.itertuples(index=False, name=None))
users = {n for n, d in B.nodes(data=True) if d["bipartite"] == "user"}
tags = set(B) - users
print("Is bipartite:", nx.is_bipartite(B), f"| users: {len(users):,}, hashtags: {len(tags):,}, edges: {B.number_of_edges():,}")
print(f"nx.density (wrong formula here): {nx.density(B):.5f}")
print(f"bipartite.density (|E| / (|users| × |hashtags|)): {bipartite.density(B, users):.5f}")

bridge = pd.Series({u: B.degree(u) for u in users}).sort_values(ascending=False)
print("Users who use the most distinct hashtags (bridges between topic spaces):", bridge.head(6).to_dict())

Two lessons:
- Edges in a bipartite graph can only go **between** the sets, so the number of possible edges is |users| × |hashtags|, not n(n−1)/2. `nx.density` uses the wrong denominator; `bipartite.density` uses the right one.
- Keep the graph bipartite while users and hashtags mean different things. Projecting it to a user–user or hashtag–hashtag graph is possible later, but it is an analytical decision, not a default.

---
## 4. A directed network at scale: email-Eu-core  *(slides 24–34)*
**Dataset:** anonymised email network of a large European research institution, published by the Stanford Network Analysis Project (SNAP). An edge u → v means *person u sent at least one email to person v*. Each person belongs to one of 42 departments. SNAP reports 1,005 people and 25,571 edges.

Email is not social media, but it is the cleanest public example of a **directed interaction network with node attributes**, and it behaves very differently from the airline mentions.

In [ ]:
EMAIL_URL = "https://snap.stanford.edu/data/email-Eu-core.txt.gz"
DEPT_URL = "https://snap.stanford.edu/data/email-Eu-core-department-labels.txt.gz"
email_edges = pd.read_csv(EMAIL_URL, sep=" ", header=None, names=["source", "target"])
dept = pd.read_csv(DEPT_URL, sep=" ", header=None, names=["node", "department"]).set_index("node")["department"]
print(email_edges.shape, "| departments:", dept.nunique())
display(email_edges.head(3))

E = nx.from_pandas_edgelist(email_edges, "source", "target", create_using=nx.DiGraph())
E.add_nodes_from(dept.index)                                       # people who never emailed are still people
nx.set_node_attributes(E, dept.to_dict(), "department")
loops = nx.number_of_selfloops(E)
E.remove_edges_from(nx.selfloop_edges(E))
print(f"Removed {loops} self-loops (emails to oneself)")
print(f"n = {E.number_of_nodes():,}, m = {E.number_of_edges():,}, density = {nx.density(E):.4f}")

**Best practice:** decide what to do with self-loops and isolated nodes, and say so. A person who never sent or received email is still part of the organisation, so we keep isolates; an email to yourself is not an interaction, so we drop self-loops.

In [ ]:
deg = pd.DataFrame({"in": dict(E.in_degree()), "out": dict(E.out_degree())})
print(deg.describe().loc[["mean", "50%", "max"]].round(1).T)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, color in [(axes[0], "in", ORANGE), (axes[1], "out", BLUE)]:
    ax.hist(deg[col], bins=40, color=color)
    ax.set_xlabel(f"{col}-degree"); ax.set_ylabel("people"); ax.set_title(f"{col}-degree distribution")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(deg["out"], deg["in"], s=8, alpha=0.4, color=BLUE)
lim = deg.values.max()
ax.plot([0, lim], [0, lim], color=GREY, lw=1, ls="--")
ax.set_xlabel("out-degree (people emailed)"); ax.set_ylabel("in-degree (people who emailed them)")
ax.set_title(f"in vs out (correlation {deg['in'].corr(deg['out']):.2f})")
plt.tight_layout(); plt.show()
print(f"Correlation between in- and out-degree: {deg['in'].corr(deg['out']):.2f}")

Unlike the airline mentions (where one side only receives), here we can check whether in- and out-degree move together: a high correlation means people who write to many colleagues are also written to by many. Points far below the diagonal are "broadcasters" (many sent, few received); points far above are "receivers".

### 4.1 Reciprocity and connectivity

In [ ]:
r_email = nx.reciprocity(E)
print(f"Reciprocity (email): {r_email:.2f}   vs   airline mentions: {nx.reciprocity(M):.3f}")
print(f"→ {r_email:.0%} of directed email links are matched by a link in the opposite direction.")
wcc = sorted(nx.weakly_connected_components(E), key=len, reverse=True)
scc = sorted(nx.strongly_connected_components(E), key=len, reverse=True)
print(f"Weakly connected components: {len(wcc)} (largest: {len(wcc[0])} people)")
print(f"Strongly connected components: {len(scc)} (largest: {len(scc[0])} people)")

- **Reciprocity**: compare the email value with the airline mentions. The same measure tells two very different stories, *conversation* vs *one-way attention*.
- **Weakly connected**: you can reach everyone if you ignore direction. **Strongly connected**: you can reach everyone following the arrows. The difference shows people who only send or only receive.

### 4.2 Density is easy to misuse  *(slides 33–34)*
Let's compute the density of each department's internal email network and compare it with its size.

In [ ]:
rows = []
for d_id, members in dept.groupby(dept).groups.items():
    sub = E.subgraph(members)
    if sub.number_of_nodes() >= 3:
        rows.append({"department": int(d_id), "people": sub.number_of_nodes(), "internal edges": sub.number_of_edges(),
                     "density": nx.density(sub)})
dept_stats = pd.DataFrame(rows).sort_values("people", ascending=False)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.scatter(dept_stats["people"], dept_stats["density"], color=BLUE, s=30)
ax.axhline(nx.density(E), color=ORANGE, ls="--", lw=1, label=f"whole organisation ({nx.density(E):.3f})")
ax.set_xscale("log"); ax.set_xlabel("department size (people, log)"); ax.set_ylabel("internal density")
ax.set_title("Smaller departments look 'denser'"); ax.legend()
plt.tight_layout(); plt.show()
display(dept_stats.head(5).round(3))

within = sum(1 for u, v in E.edges() if E.nodes[u]["department"] == E.nodes[v]["department"])
print(f"Share of emails sent within the same department: {within / E.number_of_edges():.0%}")

Departments are much denser than the organisation as a whole, and the smallest departments tend to be the densest. This does **not** mean small departments are "better" communities (slide 34):
1. **Size matters:** a 5-person team needs only 20 directed edges to be complete; 1,000 people need almost a million.
2. **Edge definition matters:** email density, follower density and hashtag density measure different relationships.
3. **Density ≠ quality:** high density can also mean a closed group that rarely talks to anyone else.

Comparing densities is only fair between networks with similar size and the same edge definition. Otherwise, compare something size-independent, such as the average degree or the share of within-group edges.

---
## 5. Good practice for drawing and sharing networks  *(slide 40)*
Layout algorithms choose positions; **position itself has no meaning**. A spring layout places connected nodes close together, but distances are approximate and change with the random seed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
largest_dept = dept_stats.iloc[0]["department"]
S = E.subgraph(dept[dept == largest_dept].index).copy()
S.remove_nodes_from(list(nx.isolates(S)))

nx.draw(S, ax=axes[0], node_size=30, width=0.3, arrows=False)
axes[0].set_title("Default drawing: a hairball")

indeg_s = dict(S.in_degree())
p = nx.spring_layout(S, seed=SEED, k=0.5)
top = sorted(indeg_s, key=indeg_s.get, reverse=True)[:5]
nx.draw_networkx_edges(S, p, ax=axes[1], edge_color="#d9d8d4", width=0.4, arrows=False)
nx.draw_networkx_nodes(S, p, ax=axes[1], node_size=[10 + 6 * indeg_s[n] for n in S],
                       node_color=[ORANGE if n in top else BLUE for n in S], alpha=0.85)
axes[1].set_title(f"Department {int(largest_dept)}: size = in-degree, top-5 in orange, fixed seed")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()
display(pd.DataFrame({"person": top, "in-degree": [indeg_s[n] for n in top]}).set_index("person").T)

| Do | Don't |
|---|---|
| Fix the layout seed (`seed=42`) so the picture is reproducible | Read meaning into exact positions or distances |
| Map **one** measure to node size and say which (title or legend) | Use size, colour and labels for three unexplained things |
| Filter: show the component, period or weight threshold that answers the question | Draw 10,000 nodes and call it insight |
| Highlight only the few nodes you talk about (and put their numbers in a table if labels would overlap) | Label every node |
| Report the numbers (degree table, density) next to the picture | Let the picture replace the measures |

For large graphs, export to **Gephi** (free desktop tool) and do the layout there. GEXF keeps node and edge attributes:

In [ ]:
# GEXF/GraphML only store simple types (str, int, float, bool): convert timestamps to text first
M_export = M.copy()
for _, _, d in M_export.edges(data=True):
    d["first_seen"], d["last_seen"] = str(d["first_seen"]), str(d["last_seen"])
nx.write_gexf(M_export, "airline_mention_network.gexf")
nx.write_gexf(HT, "hashtag_cooccurrence.gexf")
mention_edges.to_csv("airline_mention_edges.csv", index=False)       # the edge list is the most portable format
print("Saved: airline_mention_network.gexf, hashtag_cooccurrence.gexf, airline_mention_edges.csv (Colab: Files panel)")